In [1]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

/var/folders/x1/8d0k5jps45v87fgxrqdbfhqr0000gn/T/ipykernel_39823/885881214.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
/Users/abhinashsamal/Documents/AI-Projects/rag-project/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from pathlib import Path

def process_all_pdfs(pdf_directory):
    pdf_dir_path = Path(pdf_directory)
    pdf_files = list(pdf_dir_path.glob("*.pdf"))
    print(f"Found {len(pdf_files)} PDF files")
    
    all_documents = []
    
    for pdf_file in pdf_files:
        print(f"Processing: {pdf_file.name}")
        loader = PyPDFLoader(str(pdf_file))
        documents = loader.load()
        
        # Add additional metadata
        for doc in documents:
            doc.metadata["source_file"] = pdf_file.name
            doc.metadata["file_type"] = "pdf"
            
        all_documents.extend(documents)
        print(f"Loaded {len(documents)} pages from {pdf_file.name}")
        
    return all_documents

# Call the function to load documents
all_pdf_documents = process_all_pdfs("../data")

Ignoring missing Ascii85 end marker.


Found 1 PDF files
Processing: dokumen.pub_system-design-interview-an-insiders-guide-volume-2-1736049119-9781736049112.pdf


Ignoring missing Ascii85 end marker.


Loaded 427 pages from dokumen.pub_system-design-interview-an-insiders-guide-volume-2-1736049119-9781736049112.pdf


In [3]:
### Text splitting get into chunks

def split_documents(documents, chunk_size=1000, chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    # Optional preview of the first chunk
    if split_docs:
        print("\nFirst chunk preview:")
        print("Content:", split_docs[0].page_content[:200])
        print("Metadata:", split_docs[0].metadata)
        
    return split_docs

In [4]:
# Pass your loaded documents to create the chunks
chunks = split_documents(all_pdf_documents)


Split 427 documents into 0 chunks


In [5]:
chunks

[]

In [6]:
all_pdf_documents

[Document(metadata={'producer': 'PyPDF', 'creator': 'vFlat', 'creationdate': '', 'source': '../data/dokumen.pub_system-design-interview-an-insiders-guide-volume-2-1736049119-9781736049112.pdf', 'total_pages': 427, 'page': 0, 'page_label': '1', 'source_file': 'dokumen.pub_system-design-interview-an-insiders-guide-volume-2-1736049119-9781736049112.pdf', 'file_type': 'pdf'}, page_content=''),
 Document(metadata={'producer': 'PyPDF', 'creator': 'vFlat', 'creationdate': '', 'source': '../data/dokumen.pub_system-design-interview-an-insiders-guide-volume-2-1736049119-9781736049112.pdf', 'total_pages': 427, 'page': 1, 'page_label': '2', 'source_file': 'dokumen.pub_system-design-interview-an-insiders-guide-volume-2-1736049119-9781736049112.pdf', 'file_type': 'pdf'}, page_content=''),
 Document(metadata={'producer': 'PyPDF', 'creator': 'vFlat', 'creationdate': '', 'source': '../data/dokumen.pub_system-design-interview-an-insiders-guide-volume-2-1736049119-9781736049112.pdf', 'total_pages': 427, 

In [7]:
# Check how many documents have actual text
empty_docs = [doc for doc in all_pdf_documents if not doc.page_content.strip()]
print(f"Total documents: {len(all_pdf_documents)}")
print(f"Empty documents: {len(empty_docs)}")

# Inspect the first non-empty page (if any)
for i, doc in enumerate(all_pdf_documents[:5]):
    print(f"Page {i} content length: {len(doc.page_content)}")

Total documents: 427
Empty documents: 427
Page 0 content length: 0
Page 1 content length: 0
Page 2 content length: 0
Page 3 content length: 0
Page 4 content length: 0


In [8]:
from pdf2image import convert_from_path
import pytesseract
from langchain_core.documents import Document

pdf_path = "/Users/abhinashsamal/Documents/AI-Projects/rag-project/data/dokumen.pub_system-design-interview-an-insiders-guide-volume-2-1736049119-9781736049112.pdf"

ocr_documents = []

# Test with first 3 pages (first_page=1, last_page=3) to verify before running all 427 pages
images = convert_from_path(pdf_path, first_page=1, last_page=3)

for i, image in enumerate(images):
    print(f"Running OCR on page {i + 1}...")
    text = pytesseract.image_to_string(image)
    
    if text.strip():
        ocr_documents.append(
            Document(
                page_content=text,
                metadata={
                    "source_file": "system-design-v2.pdf",
                    "page": i + 1,
                    "file_type": "pdf"
                }
            )
        )

print(f"\nExtracted {len(ocr_documents)} pages successfully.")

# Pass to your split_documents function from the tutorial:
chunks = split_documents(ocr_documents)

Running OCR on page 1...
Running OCR on page 2...
Running OCR on page 3...

Extracted 3 pages successfully.
Split 3 documents into 3 chunks

First chunk preview:
Content: SYSTEM
DESIGN
INTERVIEW ©

ofsssale

AN INSIDER'S GUIDE

| [es
§) ByteByteGo AlexXu&SahnlLam |
Metadata: {'source_file': 'system-design-v2.pdf', 'page': 1, 'file_type': 'pdf'}


In [9]:
chunks


[Document(metadata={'source_file': 'system-design-v2.pdf', 'page': 1, 'file_type': 'pdf'}, page_content="SYSTEM\nDESIGN\nINTERVIEW ©\n\nofsssale\n\nAN INSIDER'S GUIDE\n\n| [es\n§) ByteByteGo AlexXu&SahnlLam |"),
 Document(metadata={'source_file': 'system-design-v2.pdf', 'page': 2, 'file_type': 'pdf'}, page_content='System Design Interview\n\nAn Insider’s Guide\nVolume 2\n\nAlex Xu | Sahn Lam\n\nf) ByteByteGo'),
 Document(metadata={'source_file': 'system-design-v2.pdf', 'page': 3, 'file_type': 'pdf'}, page_content="SYSTEM DESIGN INTERVIEW - AN INSIDER'S GUIDE (VOLUME. 2)\nCopyright ©2022 Byte Code LLC\n\nAll rights reserved. This book or any portion thereof may not be reproduced or used in\nany manner whatsoever without the express written permission of the publisher except\nfor the use of brief quotations in a book review.\n\nJoin the community\n\nWe created a members-only Discord group. It is designed for community discussions On\n\nthe following topics:\n\n+ System design fundamental

In [10]:
from pdf2image import convert_from_path
import pytesseract
from langchain_core.documents import Document

pdf_path = "/Users/abhinashsamal/Documents/AI-Projects/rag-project/data/dokumen.pub_system-design-interview-an-insiders-guide-volume-2-1736049119-9781736049112.pdf"

print("Converting first 100 PDF pages to images...")
# Convert pages 1 to 100
images = convert_from_path(pdf_path, first_page=1, last_page=100)

ocr_documents = []

print(f"Running OCR on {len(images)} pages...")
for i, image in enumerate(images):
    # Print progress every 10 pages so you know it is running
    if (i + 1) % 10 == 0 or i == 0:
        print(f"Processing page {i + 1}/100...")
        
    text = pytesseract.image_to_string(image)
    
    if text.strip():
        ocr_documents.append(
            Document(
                page_content=text,
                metadata={
                    "source_file": "system-design-v2.pdf",
                    "page": i + 1,
                    "file_type": "pdf"
                }
            )
        )

print(f"\nCompleted! Extracted text from {len(ocr_documents)} pages.")

# Split into chunks as shown at 1:00:11 in the crash course
chunks = split_documents(ocr_documents)

Converting first 100 PDF pages to images...
Running OCR on 100 pages...
Processing page 1/100...
Processing page 10/100...
Processing page 20/100...
Processing page 30/100...
Processing page 40/100...
Processing page 50/100...
Processing page 60/100...
Processing page 70/100...
Processing page 80/100...
Processing page 90/100...
Processing page 100/100...

Completed! Extracted text from 99 pages.
Split 99 documents into 208 chunks

First chunk preview:
Content: SYSTEM
DESIGN
INTERVIEW ©

ofsssale

AN INSIDER'S GUIDE

| [es
§) ByteByteGo AlexXu&SahnlLam |
Metadata: {'source_file': 'system-design-v2.pdf', 'page': 1, 'file_type': 'pdf'}


In [11]:
chunks

[Document(metadata={'source_file': 'system-design-v2.pdf', 'page': 1, 'file_type': 'pdf'}, page_content="SYSTEM\nDESIGN\nINTERVIEW ©\n\nofsssale\n\nAN INSIDER'S GUIDE\n\n| [es\n§) ByteByteGo AlexXu&SahnlLam |"),
 Document(metadata={'source_file': 'system-design-v2.pdf', 'page': 2, 'file_type': 'pdf'}, page_content='System Design Interview\n\nAn Insider’s Guide\nVolume 2\n\nAlex Xu | Sahn Lam\n\nf) ByteByteGo'),
 Document(metadata={'source_file': 'system-design-v2.pdf', 'page': 3, 'file_type': 'pdf'}, page_content="SYSTEM DESIGN INTERVIEW - AN INSIDER'S GUIDE (VOLUME. 2)\nCopyright ©2022 Byte Code LLC\n\nAll rights reserved. This book or any portion thereof may not be reproduced or used in\nany manner whatsoever without the express written permission of the publisher except\nfor the use of brief quotations in a book review.\n\nJoin the community\n\nWe created a members-only Discord group. It is designed for community discussions On\n\nthe following topics:\n\n+ System design fundamental

In [12]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [13]:
class EmbeddingManager:
    """Handles document embedding generation using sentence transformers"""
    
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        self.model_name = model_name
        self.model = None
        self._load_model()
        
    def _load_model(self):
        """Load the sentence transformer model"""
        print(f"Loading embedding model: {self.model_name}")
        self.model = SentenceTransformer(self.model_name)
        self.dimension = self.model.get_embedding_dimension()
        print(f"Model loaded successfully. Dimension: {self.dimension}")
        
    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """Generate embeddings for a list of texts"""
        embeddings = self.model.encode(texts, show_progress_bar=True)
        return embeddings

# Initialize the manager
embedding_manager = EmbeddingManager()

Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 9119.78it/s]


Model loaded successfully. Dimension: 384


In [14]:
embedding_manager

In [15]:
import os

class VectorStore:
    """Persistent ChromaDB vector store wrapper"""
    
    def __init__(self, collection_name: str = "pdf_documents", persistent_dir: str = "../data/chroma_db"):
        self.collection_name = collection_name
        self.persistent_dir = persistent_dir
        self.client = None
        self.collection = None
        self._init_store()
        
    def _init_store(self):
        """Initialize persistent Chroma client and collection"""
        os.makedirs(self.persistent_dir, exist_ok=True)
        self.client = chromadb.PersistentClient(path=self.persistent_dir)
        self.collection = self.client.get_or_create_collection(
            name=self.collection_name,
            metadata={"hnsw:space": "cosine"}
        )
        print(f"Vector store ready. Existing documents in collection: {self.collection.count()}")

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """Add chunks and precomputed embeddings to ChromaDB"""
        ids = []
        embedding_list = []
        metadatas = []
        document_texts = []
        
        for doc, emb in zip(documents, embeddings):
            doc_id = str(uuid.uuid4())
            ids.append(doc_id)
            embedding_list.append(emb.tolist())
            document_texts.append(doc.page_content)
            
            # Prepare metadata clean of unsupported nested structures
            clean_metadata = {k: str(v) for k, v in doc.metadata.items()}
            metadatas.append(clean_metadata)
            
        self.collection.add(
            ids=ids,
            embeddings=embedding_list,
            documents=document_texts,
            metadatas=metadatas
        )
        print(f"Successfully added {len(ids)} documents. Total in collection: {self.collection.count()}")

# Initialize the store
vector_store = VectorStore()

Vector store ready. Existing documents in collection: 416


In [16]:
# Extract text contents from your generated chunks
texts = [doc.page_content for doc in chunks]

# Generate embeddings (shows progress bar)
chunk_embeddings = embedding_manager.generate_embeddings(texts)

# Store chunks and their embeddings into ChromaDB
vector_store.add_documents(documents=chunks, embeddings=chunk_embeddings)

Batches: 100%|██████████| 7/7 [00:02<00:00,  2.86it/s]


Successfully added 208 documents. Total in collection: 624


In [17]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""
    
    def __init__(self, vector_store, embedding_manager):
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager
        
    def retrieve(self, query: str, top_k: int = 3, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """Retrieve relevant documents for a search query"""
        
        # 1. Convert the text query into vector embeddings
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        
        # 2. Query the ChromaDB collection
        results = self.vector_store.collection.query(
            query_embeddings=[query_embedding.tolist()],
            n_results=top_k
        )
        
        retrieved_docs = []
        
        # 3. Process and format the results
        if results['documents'] and len(results['documents']) > 0:
            # Chroma returns a list of lists, so we access index 0
            for doc_id, doc_text, metadata, distance in zip(
                results['ids'][0],
                results['documents'][0],
                results['metadatas'][0],
                results['distances'][0]
            ):
                # Calculate similarity score (1 - distance)
                similarity_score = 1.0 - distance
                
                # Filter out weak matches based on the threshold
                if similarity_score >= score_threshold:
                    retrieved_docs.append({
                        "id": doc_id,
                        "content": doc_text,
                        "metadata": metadata,
                        "similarity_score": similarity_score
                    })
                    
        return retrieved_docs

In [18]:
# Initialize the retriever using the instances you created earlier
rag_retriever = RAGRetriever(
    vector_store=vector_store, 
    embedding_manager=embedding_manager
)

# Test it with a query relevant to your PDF
query = "What is a distributed system?"
results = rag_retriever.retrieve(query=query, top_k=3)

# Print the top matches
print(f"Found {len(results)} relevant chunks for query: '{query}'\n")

for i, doc in enumerate(results):
    print(f"--- Top Match {i+1} (Score: {doc['similarity_score']:.4f}) ---")
    print(f"Page: {doc['metadata'].get('page', 'Unknown')}")
    print(doc['content'][:300] + "...\n")

Batches: 100%|██████████| 1/1 [00:00<00:00,  6.63it/s]

Found 3 relevant chunks for query: 'What is a distributed system?'

--- Top Match 1 (Score: 0.5779) ---
Page: 5
This book also provides a step-by-step framework for how to tackle a system design
question. It provides many examples to illustrate the systematic approach, with detailed
steps that you can follow. With regular practice, you will be well-equipped to tackle
system design interview questions.

This b...

--- Top Match 2 (Score: 0.5779) ---
Page: 5
This book also provides a step-by-step framework for how to tackle a system design
question. It provides many examples to illustrate the systematic approach, with detailed
steps that you can follow. With regular practice, you will be well-equipped to tackle
system design interview questions.

This b...

--- Top Match 3 (Score: 0.5779) ---
Page: 5
This book also provides a step-by-step framework for how to tackle a system design
question. It provides many examples to illustrate the systematic approach, with detailed
steps that you can

In [19]:
# Initialize the retriever using the instances you created earlier
rag_retriever = RAGRetriever(
    vector_store=vector_store, 
    embedding_manager=embedding_manager
)

# Test it with a query relevant to your PDF
query = "proximity service"
results = rag_retriever.retrieve(query=query, top_k=3)

# Print the top matches
print(f"Found {len(results)} relevant chunks for query: '{query}'\n")

for i, doc in enumerate(results):
    print(f"--- Top Match {i+1} (Score: {doc['similarity_score']:.4f}) ---")
    print(f"Page: {doc['metadata'].get('page', 'Unknown')}")
    print(doc['content'][:300] + "...\n")

Batches: 100%|██████████| 1/1 [00:00<00:00, 18.45it/s]

Found 3 relevant chunks for query: 'proximity service'

--- Top Match 1 (Score: 0.6813) ---
Page: 4
Contents

Foreword iii
Acknowledgements v
Chapter1 Proximity Service LL
Chapter 2 Nearby Friends 35
Chapter 3 Google Maps 59
Chapter4 Distributed Message Queue 91
Chapter5 Metrics Monitoring and Alerting System 131
Chapter6 Ad Click Event Aggregation 159
Chapter 7 Hotel Reservation System 195
Chapte...

--- Top Match 2 (Score: 0.6813) ---
Page: 4
Contents

Foreword iii
Acknowledgements v
Chapter1 Proximity Service LL
Chapter 2 Nearby Friends 35
Chapter 3 Google Maps 59
Chapter4 Distributed Message Queue 91
Chapter5 Metrics Monitoring and Alerting System 131
Chapter6 Ad Click Event Aggregation 159
Chapter 7 Hotel Reservation System 195
Chapte...

--- Top Match 3 (Score: 0.6813) ---
Page: 4
Contents

Foreword iii
Acknowledgements v
Chapter1 Proximity Service LL
Chapter 2 Nearby Friends 35
Chapter 3 Google Maps 59
Chapter4 Distributed Message Queue 91
Chapter5 Metrics Monitoring and Alertin

In [31]:
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv

load_dotenv()

groq_api_key = os.getenv("groq_api_key")

llm = ChatGroq(groq_api_key=groq_api_key, model_name="openai/gpt-oss-20b", temperature=0.1, max_tokens=1024)

In [32]:
## 2. Simple RAG function: retrieve context + generate response
def rag_simple(query, retriever, llm, top_k=3):
    ## retrieve the context
    results = retriever.retrieve(query, top_k=top_k)
    
    # Check if any results were found
    if not results:
        return "No relevant context found in the database."
        
    # Combine the text content from the retrieved chunks
    context_text = "\n\n".join([doc["content"] for doc in results])
    
    # Build the prompt instructing the LLM to use the context
    prompt = f"""Use the following pieces of retrieved context to answer the user's question. 
    If you don't know the answer based on the context, just say that you don't know, don't try to make up an answer.
    
    Context:
    {context_text}
    
    Question: {query}
    
    Answer:"""
    
    # Send the prompt to the Groq LLM
    response = llm.invoke(prompt)
    
    # Return just the text content of the response
    return response.content

In [35]:
# Define your question
user_query = "What is proximity"

# Call the simple RAG function
answer = rag_simple(
    query=user_query, 
    retriever=rag_retriever, # Make sure this matches the variable name of your initialized RAGRetriever
    llm=llm
)

print(answer)

Batches: 100%|██████████| 1/1 [00:01<00:00,  1.28s/it]


Proximity, in the context of the material you provided, refers to the geographic closeness between a user and other entities (such as businesses or friends). It is the concept that underlies services that deliver information about nearby places or people, where the “nearby” aspect is determined by how close those entities are to the user’s current location.


In [36]:
def rag_advanced(query, retriever, llm, top_k=3, min_score=0.1, return_context=False):
    """Advanced RAG function returning sources, confidence scores, and structured output."""
    
    # 1. Retrieve the context documents
    results = retriever.retrieve(query, top_k=top_k)
    
    # Check if results are completely empty
    if not results:
        return {
            "answer": "No relevant context found to answer the question.",
            "sources": [],
            "confidence": 0.0,
            "context": ""
        }

    context = ""
    sources = []
    total_score = 0
    
    # 2. Iterate through results and extract metadata & context
    for doc in results:
        score = doc.get("similarity_score", 0)
        
        # Filter based on the minimum threshold
        if score >= min_score:
            context += doc["content"] + "\n\n"
            total_score += score
            
            # Extract source information (Metadata)
            source_info = {
                "source": doc["metadata"].get("source_file", "Unknown"),
                "page": doc["metadata"].get("page", "Unknown"),
                "score": round(score, 4),
                "preview": doc["content"][:300] + "..." # First 300 characters for preview
            }
            sources.append(source_info)
            
    # Check if anything passed the minimum score threshold
    if not context:
         return {
            "answer": "No relevant context found above the minimum score threshold.",
            "sources": [],
            "confidence": 0.0,
            "context": ""
        }
         
    # 3. Calculate average confidence score
    confidence = round(total_score / len(sources), 4) if sources else 0.0

    # 4. Generate the answer using Groq LLM
    prompt = f"""Use the following context to answer the question concisely.
    Context:
    {context}
    
    Question: {query}
    
    Answer:"""
    
    response = llm.invoke(prompt)
    
    # 5. Prepare the structured final output
    output = {
        "answer": response.content,
        "sources": sources,
        "confidence": confidence
    }
    
    if return_context:
        output["context"] = context
        
    return output

In [38]:
# Call the advanced function
result = rag_advanced(
    query="explain the quadtree", 
    retriever=rag_retriever, 
    llm=llm,
    top_k=3,
    min_score=0.1 # Filters out bad matches
)

# Print the final structured output
print("🤖 FINAL ANSWER:")
print("-" * 60)
print(result["answer"])
print("\n📊 CONFIDENCE SCORE:", result["confidence"])

print("\n📚 SOURCES USED:")
print("-" * 60)
for i, source in enumerate(result["sources"]):
    print(f"\nSource {i+1}: {source['source']} (Page {source['page']})")
    print(f"Similarity Score: {source['score']}")
    print(f"Preview: {source['preview']}")

Batches: 100%|██████████| 1/1 [00:00<00:00,  1.93it/s]


🤖 FINAL ANSWER:
------------------------------------------------------------
A **quadtree** is a spatial index that recursively partitions a 2‑D area into four equal quadrants.  
- **Root node** represents the entire region (here, the whole world).  
- If a node contains more than a set threshold (e.g., 100 businesses), it is split into four child nodes: **NW, NE, SW, SE**.  
- Each child node is then examined; if it still exceeds the threshold, it is subdivided again.  
- The process stops when every leaf node holds at most the threshold number of items.  

In the example, the world has 200 million businesses. The root splits into four quadrants with 40 m, 30 m, 70 m, and 60 m businesses respectively. Each of these is further subdivided until every leaf node contains ≤ 100 businesses. This hierarchical structure allows efficient spatial queries such as nearest‑neighbor or range searches.

📊 CONFIDENCE SCORE: 0.6273

📚 SOURCES USED:
-----------------------------------------------------

In [39]:
# Call the advanced function
result = rag_advanced(
    query="explain the geohash", 
    retriever=rag_retriever, 
    llm=llm,
    top_k=3,
    min_score=0.1 # Filters out bad matches
)

# Print the final structured output
print("🤖 FINAL ANSWER:")
print("-" * 60)
print(result["answer"])
print("\n📊 CONFIDENCE SCORE:", result["confidence"])

print("\n📚 SOURCES USED:")
print("-" * 60)
for i, source in enumerate(result["sources"]):
    print(f"\nSource {i+1}: {source['source']} (Page {source['page']})")
    print(f"Similarity Score: {source['score']}")
    print(f"Preview: {source['preview']}")

Batches: 100%|██████████| 1/1 [00:00<00:00,  1.56it/s]


🤖 FINAL ANSWER:
------------------------------------------------------------
Geohashing is a spatial encoding technique that turns a geographic location into a short string of letters and digits. It works by flattening the Earth’s surface into a grid, then recursively subdividing that grid into smaller sub‑grids (square or rectangular). Each subdivision is labeled with a digit from 0 to 3, and the sequence of these digits (often converted to base‑32 characters) forms the geohash. The resulting string uniquely identifies a specific area, with longer strings giving finer precision.

📊 CONFIDENCE SCORE: 0.7873

📚 SOURCES USED:
------------------------------------------------------------

Source 1: system-design-v2.pdf (Page 70)
Similarity Score: 0.7873
Preview: Geohashing

Geohashing is an encoding system that encodes a geographic area into a short string of
letters and digits. At its core, it depicts the earth as a flattened surface and recursively
divides the grids into sub-grids, which